# OPAA di-Zn phosphotriesterase theozyme — TS pipeline (command / sbatch generator)

This notebook is a **command generator**, in the lab house style: each driver cell
builds the `python -m quantum_engine.cli ...` command(s) and writes them to a
commands file, then a `[SETUP BATCH JOBS]` block submits a SLURM array. **No heavy
compute runs in a cell.** Knobs (`MODEL`/`ENGINE`/`CHARGE`/`HEAD`/`RIGOR`/...) are at
the top of each step. The OPAA di-Zn site is only the *test case* — nothing here is
hardcoded to it; the reaction lives in `spec.yaml`.

See `docs/ts_workflow.md` for the canonical core + entry-point decision tree.

## INIT — project, directories, helpers (run once)

In [ ]:
import os, sys, json, textwrap
from pathlib import Path

# ── project ──────────────────────────────────────────────────────────
PROJECT_NAME = "opaa_theozyme_ts"
HOME_DIR     = Path.home()
THEOZYME_DIR = Path("/home/woodbuse/for/antonia/opaa_theozyme")   # input PDBs live here
REPO         = Path("/home/woodbuse/codebase_projects/quantum_cowboy_biochemistry")

_WORKING_DIR_OVERRIDE = None     # default: alongside this notebook
_OUTPUT_DIR_OVERRIDE  = None

WORKING_SUBDIRS = ["INPUTS", "CMDS", "LOGS", "SPECS"]
OUTPUT_SUBDIRS  = ["PROTONATE", "TS_ENTRY", "ORCA", "REPORTS"]

# ── vendored notebook helpers (resolve dirs + SLURM submit) ──────────
sys.path.insert(0, str(REPO / "notebooks" / "lib"))
import notebook_core as nb

WORKING_DIR = nb.resolve_working_dir(_WORKING_DIR_OVERRIDE)
OUTPUT_DIR  = nb.resolve_output_dir(WORKING_DIR, _OUTPUT_DIR_OVERRIDE)
nb.setup_directories(WORKING_DIR, WORKING_SUBDIRS, export_globals=True, globals_dict=globals())
nb.setup_directories(OUTPUT_DIR,  OUTPUT_SUBDIRS,  export_globals=True, globals_dict=globals())
nb.print_initialization(WORKING_DIR, OUTPUT_DIR, project_name=PROJECT_NAME)

# ── container selection (POLAR → main sif; UMA → sidecar) ────────────
QC_DIR = "/net/software/containers/users/woodbuse/quantum_chem"
def newest(glob_):
    import glob
    hits = sorted(glob.glob(glob_), reverse=True)
    return hits[0] if hits else None
MAIN_SIF = newest(f"{QC_DIR}/quantum_chem-*.sif")
UMA_SIF  = newest(f"{QC_DIR}/uma-*.sif")

def container_for(model: str) -> str:
    return UMA_SIF if str(model).lower().startswith("uma-") else MAIN_SIF

def qcb(model, *args) -> str:
    """Build one `apptainer exec ... python -m quantum_engine.cli ...` command."""
    sif = container_for(model)
    binds = "--nv --bind /home --bind /net"
    body = " ".join(str(a) for a in args)
    return (f"apptainer exec {binds} {sif} "
            f"python -m quantum_engine.cli {body}")

def write_cmds(name, commands):
    """Write a commands file under CMDS_DIR; return its path."""
    p = Path(CMDS_DIR) / name
    p.write_text("\n".join(commands) + "\n")
    print(f"wrote {len(commands)} command(s) → {p}")
    return p

print("MAIN_SIF:", MAIN_SIF)
print("UMA_SIF :", UMA_SIF)

## Step 1 — Protonate the theozyme

Deterministic staged protonation (`qcb protonate`). Edit `INPUT_PDB`, charges, and
the ligand. Produces a protonated PDB used by every later step.

In [ ]:
# ── INPUTS / OUTPUTS / PARAMETERS ────────────────────────────────────
INPUT_PDB   = str(THEOZYME_DIR / "opaa_theozyme.pdb")   # ← your theozyme
LIGAND_CHG  = -1            # substrate net charge
MODEL       = "mace-polar-m"   # charge-aware MLFF for the di-Zn site

prot_out = str(Path(OUTPUT_DIR) / "PROTONATE")
cmd = qcb(MODEL, "protonate", INPUT_PDB, "--ligand-charge", LIGAND_CHG,
          "--outdir", prot_out)
cmds_file = write_cmds("01_protonate.cmds", [cmd])
print(cmd)

# ── [SETUP BATCH JOBS] ───────────────────────────────────────────────
# nb.submit_array_job(str(cmds_file), qtime="2:00:00", cpus_per_task=4,
#                     job_name="opaa_protonate", memory="16g",
#                     submit_file=str(Path(WORKING_DIR)/"01_protonate.sh"),
#                     logs_dir=str(LOGS_DIR), num_jobs=1, cmds_per_job=1,
#                     queue="cpu")

## Step 2 — Reaction spec

Write the reaction-agnostic `ReactionSpec` (forming/breaking bonds, the
bond-difference CV, reactive atoms) and validate it against the protonated
structure. Atom tokens: `RES:ID:NAME`, `"0:idx"` (0-based), or a 1-based serial.

In [ ]:
SPEC = textwrap.dedent("""
reaction:
  # SN2-at-phosphorus: bridging hydroxide O attacks P; leaving-group O departs.
  forming_bonds:  [["SUB:P1", "OHX:O3"]]
  breaking_bonds: [["SUB:P1", "SUB:O5"]]
  reactive_atoms: ["SUB:P1", "OHX:O3", "SUB:O5"]
  cv: {kind: bond_difference, atoms: ["SUB:P1", "OHX:O3", "SUB:O5"]}  # (P, nuc, lg)
""").strip()
spec_path = Path(SPECS_DIR) / "opaa.spec.yaml"
spec_path.write_text(SPEC + "\n")

PROT_PDB = str(Path(OUTPUT_DIR) / "PROTONATE" / "protonated.pdb")   # ← protonator output
MODEL = "mace-polar-m"
cmd = qcb(MODEL, "reaction-spec", str(spec_path), "--structure", PROT_PDB)
print(cmd, "\n\n# run this locally to confirm the spec resolves before sbatching Step 3")

## Step 3 — `ts-entry` (reactant-product, MLFF)

The canonical core: clean basins → path search → saddle → Hessian gate → IRC-like
validation. Swap `MODEL`/`RIGOR`/`PATH_METHOD`/`SADDLE_BACKEND` freely.

In [ ]:
# ── PARAMETERS ───────────────────────────────────────────────────────
MODEL          = "mace-polar-m"
HEAD           = None              # e.g. "omol" for mace-mh-1
CHARGE         = 2                 # di-Zn site net charge (set per your ledger)
SPIN           = 1
RIGOR          = "standard"        # draft | standard | publication
PATH_METHOD    = "neb"             # neb | fsm | gsm-de
SADDLE_BACKEND = "auto"

REACTANT = str(Path(OUTPUT_DIR) / "PROTONATE" / "reactant.pdb")
PRODUCT  = str(Path(OUTPUT_DIR) / "PROTONATE" / "product.pdb")
spec_path = Path(SPECS_DIR) / "opaa.spec.yaml"
ts_out = str(Path(OUTPUT_DIR) / "TS_ENTRY")

args = ["ts-entry", "--entry", "reactant-product",
        "--reaction-spec", str(spec_path),
        "--reactant", REACTANT, "--product", PRODUCT,
        "--model", MODEL, "--charge", CHARGE, "--spin", SPIN,
        "--rigor", RIGOR, "--path-method", PATH_METHOD,
        "--saddle-backend", SADDLE_BACKEND, "--outdir", ts_out]
if HEAD: args += ["--head", HEAD]
cmd = qcb(MODEL, *args)
cmds_file = write_cmds("03_ts_entry.cmds", [cmd])
print(cmd)

# ── [SETUP BATCH JOBS] — GPU for MLFF ────────────────────────────────
# nb.submit_array_job(str(cmds_file), qtime="12:00:00", cpus_per_task=2,
#                     job_name="opaa_ts_entry", memory="32g",
#                     submit_file=str(Path(WORKING_DIR)/"03_ts_entry.sh"),
#                     logs_dir=str(LOGS_DIR), num_jobs=1, cmds_per_job=1,
#                     queue="gpu", gpu_class="a4000")

## Step 4 (optional) — DFT reference via ORCA native NEB-TS

ORCA isn't in the container; `--no-execute` *prepares* the input here, then you
`sbatch` it on a node with ORCA. Swap `ENGINE_MODEL` for the functional/basis.

In [ ]:
ENGINE_MODEL = "wB97X-D3/def2-TZVP"
orca_out = str(Path(OUTPUT_DIR) / "ORCA")
spec_path = Path(SPECS_DIR) / "opaa.spec.yaml"
REACTANT = str(Path(OUTPUT_DIR) / "PROTONATE" / "reactant.pdb")
PRODUCT  = str(Path(OUTPUT_DIR) / "PROTONATE" / "product.pdb")

# Prepare the ORCA NEB-TS input (MLFF container builds it; ORCA runs host-side).
cmd_prep = qcb("mace-omol", "ts-entry", "--entry", "reactant-product",
               "--reaction-spec", str(spec_path),
               "--reactant", REACTANT, "--product", PRODUCT,
               "--engine", "orca", "--model", ENGINE_MODEL,
               "--charge", 2, "--spin", 1, "--no-execute", "--outdir", orca_out)
print(cmd_prep)
print("\n# then on an ORCA node:  cd", orca_out, "&&  $ORCA_BIN nebts.inp > nebts.out")